In [1]:
# -*- coding: utf-8 -*-
import json
import os
import requests
import pandas as pd
import numpy as np
import certifi
import string
import unicodedata
from datetime import datetime, date, timedelta
from bs4 import BeautifulSoup

from text_importer.importers.bcul.detect import detect_issues, BculIssueDir,dir2issue
from text_importer.importers.bcul.helpers import parse_date, find_mit_file, replace_alias

import pylcs

## Small tests

In [2]:
p = '/mnt/project_impresso/original/BCUL/Le_PÃ¨re_JÃ©rÃ´me/1831/01/01'
p2 = '/mnt/project_impresso/original/BCUL/Le_PÃ¨re_JÃ©rÃ´me/1831/01'

In [18]:
os.listdir('/mnt/project_impresso/original/BCUL/')

In [34]:
titles = ['Le_Touriste__La_Suisse_illustrÃ©e',
 'Le_Veveysan_1',
 'Le_messager',
 'Le_peuple_souverain',
 'Lâ\x80\x99ami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud',
 'Mercure_suisse',
 'Messager_boÃ®teux',
 'Nouvelliste_suisse']

for journal in titles:
    print(journal)
    for dir_path, dirs, files in os.walk(os.path.join('/mnt/project_impresso/original/BCUL/', journal)):
        title = journal.split('/')[-1]
        # check if we are in the directory of a (valid) issue
        #print(dir_path)
        if (
            len(files) > 1 and 
            "solr" not in dir_path and 
            len(os.listdir(os.path.dirname(dir_path))) > 1
        ):
            print(dir_path, ' : ', os.listdir(os.path.dirname(dir_path)))

Le_Touriste__La_Suisse_illustrÃ©e
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/02/15/672279  :  ['672279', '672292']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/02/15/672292  :  ['672279', '672292']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/03/15/672277  :  ['672277', '672281']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/03/15/672281  :  ['672277', '672281']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/06/15/672285  :  ['672285', '672289']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/06/15/672289  :  ['672285', '672289']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/07/15/672278  :  ['672278', '698260']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_illustrÃ©e/1880/07/15/698260  :  ['672278', '698260']
/mnt/project_impresso/original/BCUL/Le_Touriste__La_Suisse_ill

In [28]:
dirs = os.listdir('/mnt/project_impresso/original/BCUL/Gazette_des_Villages/1867/08/31')
if '.DS_Store' in dirs:
    dirs.remove('.DS_Store')
dirs

['590435']

In [6]:
FAULTY_ISSUES = ['127626', '127627', '127628', '127629', '127630', '127631', '127625']

In [17]:
for dir_path, dirs, files in os.walk(p):
    if (
        len(files) > 1 and 
        "solr" not in dir_path and 
        os.path.basename(dir_path) not in FAULTY_ISSUES
    ):
        print(type(dir_path.split('/')[-1]), os.path.basename(dir_path), ' not in FAULTY_ISSUES')

<class 'str'> 170463  not in FAULTY_ISSUES
<class 'str'> 170464  not in FAULTY_ISSUES
<class 'str'> 170465  not in FAULTY_ISSUES
<class 'str'> 170466  not in FAULTY_ISSUES
<class 'str'> 170467  not in FAULTY_ISSUES
<class 'str'> 170468  not in FAULTY_ISSUES


In [2]:
def get_date(dayofyear: str, year: int) -> datetime.date:
        """Return the date corresponding to a day of year.

        Args:
            dayofyear (str): Numbered day in a year.
            year (int): Year in question.

        Returns:
            datetime.date: Date corresponding to the day of year.
        """
        start_date = datetime(year=year, month=1, day=1)
        return start_date + timedelta(days=int(dayofyear) - 1)

In [3]:
#uri = 'https://scriptorium.bcu-lausanne.ch/api/iiif/127765/manifest'
#uri = 'https://scriptorium.bcu-lausanne.ch/api/iiif/127571/manifest'
#uri = 'https://scriptorium.bcu-lausanne.ch/api/iiif/127901/manifest'
uri = 'https://scriptorium.bcu-lausanne.ch/api/iiif/128024/manifest'
page_number = 2
r = requests.get(uri)
if r.status_code == 200:
    page_canvas = r.json()["sequences"][0]["canvases"][page_number-1]
    page_canvas_num = int(page_canvas['label'])
    print(page_canvas_num)
    if page_canvas_num!=page_number:
        print("looking for the right page in the canvases")
        page_canvas = None
        for c in r.json()["sequences"][0]["canvases"].items():
            #if 
            if c['label'] == page_number:
                page_canvas = c
                break
    if page_canvas is None:
        print(f"ME-1732-12-01-a: Could not find the iiif link for page {page_number}.")
    iiif = page_canvas["images"][0]["resource"]["@id"]
    iiif = "/".join(iiif.split("/")[:-4])
iiif

2


'https://scriptorium.bcu-lausanne.ch/api/iiif-img/659775'

## Functions

In [2]:
def modif_str(name: str) -> str:
    # Format the string name to unify the collection names
    no_space = name.replace(':', ' ').replace(' ', '_').replace('/', '')
    
    if 'ami' in no_space:
        no_space = no_space.replace("'", "’")

    return no_space.lower()

In [3]:
def lower_to_utf8(bs):
    return str(bs.lower(), 'utf8')

In [4]:
matches = {
    b'\xc3\xaa': str(b'\xc3\xaa', 'utf-8'),
    b'\xe2\x80\x93': str(b'\xe2\x80\x93', 'utf-8'),
    b'\xc3\xae': str(b'\xc3\xae', 'utf-8'),
    b'\xc3\xa9': str(b'\xc3\xa9', 'utf-8'),
    b'\xc3\xa8': str(b'\xc3\xa8', 'utf-8'),
    b'\xc3\xb4': str(b'\xc3\xb4', 'utf-8'),
    b'\xe2\x80\x99': str(b'\xe2\x80\x99', 'utf-8'),
}
chars = [b'\xc3\xaa', b'\xe2\x80\x93', b'\xc3\xae', b'\xc3\xa9', b'\xc3\xa8', b'\xc3\xb4']

## Journal list

In [ ]:
csv_path = "../text_preparation/data/sample_data/BCUL/names_and_aliases_old.csv" # TODO replace with absolute path if needed
collection_path = '/mnt/project_impresso/original/BCUL'

# read the columns of interest of the csv
columns = ['Collection name','Journal Name','Start Year (in Scriptorium)','End Year (in Scriptorium)','Data Transfer Batch (with hard disk)']
bcul_titles = pd.read_csv(csv_path, header =1, usecols=columns, index_col=False)
# filter to only keep the first batch
# TODO change here to keep batches 3 and 4
batch_1_titles = bcul_titles[bcul_titles['Data Transfer Batch (with hard disk)'] == 'Batch 1 (21/12/23)'].copy()
batch_1_titles['End Year (in Scriptorium)'] = batch_1_titles['End Year (in Scriptorium)'].astype(int)
batch_1_titles

,Collection name,Journal Name,Start Year (in Scriptorium),End Year (in Scriptorium),Data Transfer Batch (with hard disk)
0,Gazettes suisses du 18e s.,Mercure suisse,1732,1747,Batch 1 (21/12/23)
2,Gazettes suisses du 18e s.,Nouvelliste suisse,1748,1769,Batch 1 (21/12/23)
4,Gazettes suisses du 18e s.,Journal helvétique,1738,1782,Batch 1 (21/12/23)
5,Almanachs vaudois,Messager boîteux,1748,1800,Batch 1 (21/12/23)
7,Nouvelliste vaudois,L'ami de la vérité: journal du Canton de Vaud,1822,1823,Batch 1 (21/12/23)
9,Annuaire vaudois,Almanach pour le commerce,1832,1832,Batch 1 (21/12/23)
10,Journaux satiriques vaudois,Le Père Jérôme,1831,1832,Batch 1 (21/12/23)
11,Journaux satiriques vaudois,Le Charivari [1],1839,1841,Batch 1 (21/12/23)
12,Presse de la Côte vaudoise,Le Phare de Nyon,1841,1841,Batch 1 (21/12/23)
13,Presse de la Côte vaudoise,Le Phare du Léman,1841,1842,Batch 1 (21/12/23)


In [5]:
utf8_to_unicode = {}
utf8_dirs = {}
for dir_name1, dn2 in zip(os.listdir(b'/mnt/project_impresso/original/BCUL'), os.listdir(u'/mnt/project_impresso/original/BCUL')):
    utf8_to_unicode[dir_name1] = dn2
    utf8_dirs[lower_to_utf8(dir_name1)] = dir_name1

In [6]:
batch_1_titles['displayed_name_in_fs_lower'] = batch_1_titles['Journal Name'].apply(lambda x: modif_str(x))
batch_1_titles['binary_name_in_fs'] = batch_1_titles['displayed_name_in_fs_lower'].apply(lambda x: utf8_dirs[x] if x in utf8_dirs else None)
batch_1_titles['unicode_name_in_fs'] = batch_1_titles['binary_name_in_fs'].apply(lambda x: utf8_to_unicode[x] if x in utf8_to_unicode else None)
batch_1_titles.head()

,Collection name,Journal Name,Start Year (in Scriptorium),End Year (in Scriptorium),Data Transfer Batch (with hard disk),displayed_name_in_fs_lower,binary_name_in_fs,unicode_name_in_fs
0,Gazettes suisses du 18e s.,Mercure suisse,1732,1747,Batch 1 (21/12/23),mercure_suisse,b'Mercure_suisse',Mercure_suisse
2,Gazettes suisses du 18e s.,Nouvelliste suisse,1748,1769,Batch 1 (21/12/23),nouvelliste_suisse,b'Nouvelliste_suisse',Nouvelliste_suisse
4,Gazettes suisses du 18e s.,Journal helvétique,1738,1782,Batch 1 (21/12/23),journal_helvétique,b'Journal_helv\xc3\xa9tique',Journal_helvÃ©tique
5,Almanachs vaudois,Messager boîteux,1748,1800,Batch 1 (21/12/23),messager_boîteux,b'Messager_bo\xc3\xaeteux',Messager_boÃ®teux
7,Nouvelliste vaudois,L'ami de la vérité: journal du Canton de Vaud,1822,1823,Batch 1 (21/12/23),l’ami_de_la_vérité__journal_du_canton_de_vaud,b'L\xe2\x80\x99ami_de_la_v\xc3\xa9rit\xc3\xa9_...,Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud


In [7]:
titles_not_in_batch = batch_1_titles[batch_1_titles['unicode_name_in_fs'].isna()]['Journal Name']
titles_not_in_batch

42    L'Echo de Morges
Name: Journal Name, dtype: object

In [17]:
batch_1_titles

,Collection name,Journal Name,Start Year (in Scriptorium),End Year (in Scriptorium),Data Transfer Batch (with hard disk),displayed_name_in_fs_lower,binary_name_in_fs,unicode_name_in_fs,alias
0,Gazettes suisses du 18e s.,Mercure suisse,1732,1747,Batch 1 (21/12/23),mercure_suisse,b'Mercure_suisse',Mercure_suisse,ME
2,Gazettes suisses du 18e s.,Nouvelliste suisse,1748,1769,Batch 1 (21/12/23),nouvelliste_suisse,b'Nouvelliste_suisse',Nouvelliste_suisse,NS
4,Gazettes suisses du 18e s.,Journal helvétique,1738,1782,Batch 1 (21/12/23),journal_helvétique,b'Journal_helv\xc3\xa9tique',Journal_helvÃ©tique,JH
5,Almanachs vaudois,Messager boîteux,1748,1800,Batch 1 (21/12/23),messager_boîteux,b'Messager_bo\xc3\xaeteux',Messager_boÃ®teux,MB
7,Nouvelliste vaudois,L'ami de la vérité: journal du Canton de Vaud,1822,1823,Batch 1 (21/12/23),l’ami_de_la_vérité__journal_du_canton_de_vaud,b'L\xe2\x80\x99ami_de_la_v\xc3\xa9rit\xc3\xa9_...,Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud,NV
9,Annuaire vaudois,Almanach pour le commerce,1832,1832,Batch 1 (21/12/23),almanach_pour_le_commerce,b'Almanach_pour_le_commerce',Almanach_pour_le_commerce,ACI
10,Journaux satiriques vaudois,Le Père Jérôme,1831,1832,Batch 1 (21/12/23),le_père_jérôme,b'Le_P\xc3\xa8re_J\xc3\xa9r\xc3\xb4me',Le_PÃ¨re_JÃ©rÃ´me,PJ
11,Journaux satiriques vaudois,Le Charivari [1],1839,1841,Batch 1 (21/12/23),le_charivari_[1],b'Le_Charivari_[1]',Le_Charivari_[1],Charivari
12,Presse de la Côte vaudoise,Le Phare de Nyon,1841,1841,Batch 1 (21/12/23),le_phare_de_nyon,b'Le_Phare_de_Nyon',Le_Phare_de_Nyon,PDN
13,Presse de la Côte vaudoise,Le Phare du Léman,1841,1842,Batch 1 (21/12/23),le_phare_du_léman,b'Le_Phare_du_L\xc3\xa9man',Le_Phare_du_LÃ©man,PDL


## Fetching Aliases from each issue dir

Correction: There are aliases already inside each issue.

In [8]:
base_dir = '/mnt/project_impresso/original/BCUL/'

found_aliases = {
 '/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce': 'ACI',
 '/mnt/project_impresso/original/BCUL/Castigat_ridendo_mores': 'Castigat',
 '/mnt/project_impresso/original/BCUL/Croquis_vaudois': 'Croquis',
 '/mnt/project_impresso/original/BCUL/Courrier_du_LÃ©man': 'CL',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Moudon": 'FAMDE',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Nyon": 'FAN',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Payerne,_Moudon_et_Avenches": 'feuille',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Payerne": 'feuille',
 '/mnt/project_impresso/original/BCUL/Gazette_des_Villages': 'GAV',
 '/mnt/project_impresso/original/BCUL/Indicateur_de_Lausanne': 'AV',
 "/mnt/project_impresso/original/BCUL/Journal_d'Yverdon_2": 'JY2',
 '/mnt/project_impresso/original/BCUL/Journal_de_Vevey': 'JV'
}

aliases_json= {
    "Courrier_du_LÃ©man": 'CL',
    "Feuille_d'avis_de_Moudon": 'FAMDE',
    "Feuille_d'avis_de_Nyon": 'FAN',
    "Feuille_d'avis_de_Payerne,_Moudon_et_Avenches": "feuillePMA", #'feuille' in fs
    "Feuille_d'avis_de_Payerne": "feuilleP", # 'feuille' in fs
    "Gazette_des_Villages": "GAVi", # GAV
    "Journal_d'Yverdon_2": 'JY2', 
    "Journal_de_Vevey": 'JV',
    "Journal_de_vevey_et_des_Ã©trangers": 'JVE',
    "L'Observateur_du_LÃ©man": 'OBS',
    "Le_Moustique_(Vevey)": 'Moustique', 
    "Le_Phare_de_Nyon": 'PDN', 
    "Le_Phare_du_LÃ©man": 'PDL', # FAN in fs
    "Le_Touriste__La_Suisse_illustrÃ©e": "TouSuIl", #'TO-SU-IL' in fs
    "Le_Veveysan_1": 'VVS1', # VVS also for La_Veveysanne_–_La_Patrie/La_Veveysanne 
    "Le_messager": 'MESSAGER',
    "Le_peuple_souverain": 'PS'
}

In [9]:
name_to_alias = {}
name_to_type = {}

In [7]:
def check_title_alias_in_fs(title_unicode_name: str, base_dir: str = u'/mnt/project_impresso/original/BCUL') -> tuple[str, str]:
    # fetch the alias used in the filesystem for a given journal
    journal_dir = os.path.join(base_dir, title_unicode_name)
    for dir_path, dirs, files in os.walk(journal_dir):
        if len(files)>1 and 'solr' not in dir_path:
            print(f"{dir_path}: {files[:3]}")
            for file in files:
                if 'mit' in file:
                    return file.split('_')[0], file.split('.')[-1]

In [18]:
for j_name, unicode_name in zip(batch_1_titles['Journal Name'], batch_1_titles['unicode_name_in_fs']):
    # if a name already exists, for the name, use it
    if unicode_name is None:
        continue
    else:
        # otherwise check in the filesystem for the name that is used internally
        candidate_alias, file_type = check_title_alias_in_fs(unicode_name)

        if unicode_name in aliases_json:
            name_to_alias[unicode_name] = aliases_json[unicode_name]
        else:
            while candidate_alias in aliases_json.values():
                if candidate_alias[-1].isnumeric():
                    candidate_alias[-1] = str(int(candidate_alias[-1]) + 1)
                else:
                    candidate_alias = candidate_alias + '1'
                    
            name_to_alias[unicode_name] = candidate_alias
        name_to_type[unicode_name] = file_type

/mnt/project_impresso/original/BCUL/Mercure_suisse/1732/12/01/127765: ['ME_1732_12_00_0001_mit.xml', 'ME_1732_12_00.pdf', 'ME_1732_12_00_0001.jp2']
/mnt/project_impresso/original/BCUL/Nouvelliste_suisse/1748/01/01/128065: ['NS_1748_01_00_0001_mit.xml', 'NS_1748_01_00.pdf', 'NS_1748_01_00_0001.jp2']
/mnt/project_impresso/original/BCUL/Journal_helvÃ©tique/1738/01/01/127826: ['JH_1738_01_00_0001_mit.xml', 'JH_1738_01_00.pdf', 'JH_1738_01_00_0001.jp2']
/mnt/project_impresso/original/BCUL/Messager_boÃ®teux/1748/01/01/168341: ['MB_1748_00_00_0001_mit.xml', 'MB_1748_00_00.pdf', 'MB_1748_00_00_0001.jp2']
/mnt/project_impresso/original/BCUL/Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud/1822/12/18/102037: ['NV_1822_12_18_0001_mit.xml', 'NV_1822_12_18.pdf', 'NV_1822_12_18_0001.jp2']
/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce/1832/01/01/171722: ['ACI_1832_00_00_0001_mit.xml', 'ACI_1832_00_00.pdf', 'ACI_1832_00_00_0001.jp2']
/mnt/project_impresso/original/BCUL/Le_PÃ¨re_JÃ©rÃ´

In [51]:
batch_1_titles['alias'] = batch_1_titles['unicode_name_in_fs'].apply(lambda n: name_to_alias[n] if n is not None else None)
batch_1_titles['file_type'] = batch_1_titles['unicode_name_in_fs'].apply(lambda n: name_to_type[n] if n is not None else None)
batch_1_titles['access_right'] = 'open_public'

In [54]:
indexed_titles = batch_1_titles.set_index('unicode_name_in_fs')[['alias', 'file_type', 'access_right']]
indexed_titles

,alias,file_type,access_right
unicode_name_in_fs,,,
Mercure_suisse,ME,xml,open_public
Nouvelliste_suisse,NS,xml,open_public
Journal_helvÃ©tique,JH,xml,open_public
Messager_boÃ®teux,MB,xml,open_public
Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud,NV,xml,open_public
Almanach_pour_le_commerce,ACI,xml,open_public
Le_PÃ¨re_JÃ©rÃ´me,PJ,xml,open_public
Le_Charivari_[1],Charivari,xml,open_public
Le_Phare_de_Nyon,PDN,json,open_public


In [55]:
titles_as_dict = indexed_titles.to_dict('index')
titles_as_dict

{'Mercure_suisse': {'alias': 'ME',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Nouvelliste_suisse': {'alias': 'NS',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Journal_helvÃ©tique': {'alias': 'JH',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Messager_boÃ®teux': {'alias': 'MB',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Lâ\x80\x99ami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud': {'alias': 'NV',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Almanach_pour_le_commerce': {'alias': 'ACI',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Le_PÃ¨re_JÃ©rÃ´me': {'alias': 'PJ',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Le_Charivari_[1]': {'alias': 'Charivari',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Le_Phare_de_Nyon': {'alias': 'PDN',
  'file_type': 'json',
  'access_right': 'open_public'},
 'Le_Phare_du_LÃ©man': {'alias': 'PDL',
  'file_type': 'json',
  'access_right': 'open_public'},
 "J

In [56]:
access_rights_and_aliases_file = '/home/piconti/impresso-text-acquisition/text_importer/data/sample_data/BCUL/access_rights_and_aliases.json'

with open(access_rights_and_aliases_file, 'w', encoding='utf-8') as f_out:
    json.dump(titles_as_dict, f_out, ensure_ascii=False, indent=4)

## Adding new aliases

Note: This is what I seem to have used for the creation of aliases for the second batch, but I don't remember what was contained in the `missing_aliases.json` file so it might be better to simply follow the logic on the issue.
However if things below prove to be useful that's perfect.

In [22]:
def check_title_alias_in_fs_2(title_name: str, base_dir: str = '/mnt/project_impresso/original/BCUL/') -> tuple[str, str]:
    # fetch the alias used in the filesystem for a given journal
    journal_dir = os.path.join(base_dir, title_name)
    for dir_path, dirs, files in os.walk(journal_dir):
        if len(files)>1 and 'solr' not in dir_path:
            print(f"{dir_path}: {files[:3]}")
            for file in files:
                if 'mit' in file:
                    return file.split('_')[0], file.split('.')[-1]

In [ ]:
base_dir = '/mnt/project_impresso/original/BCUL/'

access_rights_and_aliases_file = '../text_preparation/data/sample_data/BCUL/bcul_aliases.json'
missing_ars = '../text_preparation/data/sample_data/BCUL/missing_aliases.json'

with open(access_rights_and_aliases_file, "rb") as f:
        ar_and_alias = json.load(f)

with open(missing_ars, "rb") as f:
        missing_aliases = json.load(f)

Find all newly added titles that need an alias

In [43]:
existing_aliases = {k: v['alias'] for k, v in ar_and_alias.items()}

In [44]:
new_alias_needed = []

In [45]:
for j_name in os.listdir(base_dir):
    if j_name not in ar_and_alias and j_name not in ['wrong_BCUL', 'OLD', '.DS_Store', 'La_Veveysanne_â\x80\x93_La_Patrie']:
        if j_name == 'La_Veveysanne__La_Patrie':
            for sub_j_name in os.listdir(os.path.join(base_dir, j_name)):
                if all([sub_j_name not in v['bin_title'] for k,v in ar_and_alias.items()]):
                    print(f"Need new alias for {sub_j_name}")
                    new_alias_needed.append(sub_j_name)
        elif all([j_name not in v['bin_title'] for k,v in ar_and_alias.items()]):
            print(f"Need new alias for {j_name}")
            new_alias_needed.append(j_name)

Need new alias for Bulletins_du_Grand_Conseil
Need new alias for Feuille_d'avis_de_Morges_1
Need new alias for Feuille_d'avis_de_Vevey_1
Need new alias for L'Echo_de_Morges
Need new alias for L'Estafette_Journal_suisse
Need new alias for La_Patrie
Need new alias for La_Veveysane
Need new alias for Le_Nouvelliste_vaudois
Need new alias for Nouvelliste_vaudois_et_journal_national_suisse
Need new alias for Repertoires_de_Bulletins_du_Grand_Conseil
Need new alias for Travaux_des_Constituantes


In [46]:
new_aliases = {}
for j_name in new_alias_needed:
    if j_name in ['La_Patrie', 'La_Veveysane']:
        candidate_alias, file_type = check_title_alias_in_fs_2(os.path.join('La_Veveysanne__La_Patrie', j_name))
    else:
        # check in the filesystem for the name that is used internally
        candidate_alias, file_type = check_title_alias_in_fs_2(j_name)
    print(f"{j_name} first candidate: {candidate_alias}")
    while candidate_alias in existing_aliases.values():
        if candidate_alias[-1].isnumeric():
            candidate_alias = f"{candidate_alias[:-1]}{str(int(candidate_alias[-1]) + 1)}"
        else:
            candidate_alias = f"{candidate_alias}1"
                
    new_aliases[j_name] = {
        'alias': candidate_alias,
        'file_type': file_type,
        'access_right': 'open_public',
    }
    existing_aliases[j_name] = candidate_alias

/mnt/project_impresso/original/BCUL/Bulletins_du_Grand_Conseil/1829/06/06/287138: ['4926576.jp2', '4926576_exif.json', '4926577.jp2']
Bulletins_du_Grand_Conseil first candidate: RN
/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Morges_1/1854/06/10/381504: ['6224375.jp2', '6224375.xml', '6224375_exif.json']
Feuille_d'avis_de_Morges_1 first candidate: FAM
/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Vevey_1/1784/01/30/257835: ['4481734.jp2', '4481734.xml.bz2', '4481734_exif.json']
Feuille_d'avis_de_Vevey_1 first candidate: FAV1
/mnt/project_impresso/original/BCUL/L'Echo_de_Morges/1883/06/13/385203: ['6239346.jp2', '6239346.xml', '6239346_exif.json']
L'Echo_de_Morges first candidate: EM
/mnt/project_impresso/original/BCUL/L'Estafette_Journal_suisse/1862/12/15/91885: ['esta_1862_12_15_0001_mit.xml', 'esta_1862_12_15.pdf', 'esta_1862_12_15_0001.jp2']
L'Estafette_Journal_suisse first candidate: esta
/mnt/project_impresso/original/BCUL/La_Veveysanne__La_Patrie/La_Patrie/1841/1

In [47]:
new_aliases

{'Bulletins_du_Grand_Conseil': {'alias': 'RN',
  'file_type': 'json',
  'access_right': 'open_public'},
 "Feuille_d'avis_de_Morges_1": {'alias': 'FAM',
  'file_type': 'json',
  'access_right': 'open_public'},
 "Feuille_d'avis_de_Vevey_1": {'alias': 'FAV1',
  'file_type': 'json',
  'access_right': 'open_public'},
 "L'Echo_de_Morges": {'alias': 'EM',
  'file_type': 'json',
  'access_right': 'open_public'},
 "L'Estafette_Journal_suisse": {'alias': 'esta',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'La_Patrie': {'alias': 'PAT',
  'file_type': 'json',
  'access_right': 'open_public'},
 'La_Veveysane': {'alias': 'VVS',
  'file_type': 'json',
  'access_right': 'open_public'},
 'Le_Nouvelliste_vaudois': {'alias': 'NV1',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Nouvelliste_vaudois_et_journal_national_suisse': {'alias': 'NV2',
  'file_type': 'xml',
  'access_right': 'open_public'},
 'Repertoires_de_Bulletins_du_Grand_Conseil': {'alias': 'RN1',
  'file_type': 'json'

In [49]:
ar_and_alias.update(new_aliases)
ar_and_alias

{'Mercure_suisse': {'alias': 'ME',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Mercure_suisse'},
 'Nouvelliste_suisse': {'alias': 'NS',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Nouvelliste_suisse'},
 'Journal_helvétique': {'alias': 'JH',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Journal_helvÃ©tique'},
 'Messager_boîteux': {'alias': 'MB',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Messager_boÃ®teux'},
 'L’ami_de_la_vérité__journal_du_Canton_de_Vaud': {'alias': 'NV',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Lâ\x80\x99ami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud'},
 'Almanach_pour_le_commerce': {'alias': 'ACI',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Almanach_pour_le_commerce'},
 'Le_Père_Jérôme': {'alias': 'PJ',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Le_PÃ¨re_JÃ©rÃ´me'},
 'Le_Chariva

In [ ]:
access_rights_and_aliases_file = '../text_preparation/data/sample_data/BCUL/access_rights_and_aliases.json'

with open(access_rights_and_aliases_file, 'w', encoding='utf-8') as f_out:
    json.dump(ar_and_alias, f_out, ensure_ascii=False, indent=4)

Note all what is below is working or debug code for the next steps of the process. 
They might be useful in the future for refinement or debug so I keep them here but they are not necessary.

## Trying detect function

In [2]:
base_dir = '/mnt/project_impresso/original/BCUL/'

In [ ]:
access_rights_and_aliases_file = '../text_preparation/data/sample_data/BCUL/access_rights_and_aliases.json'
missing_ars = '../text_preparation/data/sample_data/BCUL/missing_aliases.json'

with open(access_rights_and_aliases_file, "rb") as f:
        ar_and_alias = json.load(f)

with open(missing_ars, "rb") as f:
        missing_aliases = json.load(f)

In [4]:
ar_and_alias

{'Mercure_suisse': {'alias': 'ME',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Mercure_suisse'},
 'Nouvelliste_suisse': {'alias': 'NS',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Nouvelliste_suisse'},
 'Journal_helvétique': {'alias': 'JH',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Journal_helvÃ©tique'},
 'Messager_boîteux': {'alias': 'MB',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Messager_boÃ®teux'},
 'L’ami_de_la_vérité__journal_du_Canton_de_Vaud': {'alias': 'NV',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Lâ\x80\x99ami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud'},
 'Almanach_pour_le_commerce': {'alias': 'ACI',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Almanach_pour_le_commerce'},
 'Le_Père_Jérôme': {'alias': 'PJ',
  'file_type': 'xml',
  'access_right': 'open_public',
  'bin_title': 'Le_PÃ¨re_JÃ©rÃ´me'},
 'Le_Chariva

In [5]:
missing_aliases

{'Almanach_pour_le_commerce': ['Almanach_pour_le_commerce',
  {'alias': 'ACI', 'file_type': 'xml', 'access_right': 'open_public'}],
 'Castigat_ridendo_mores': ['Castigat_ridendo_mores',
  {'alias': 'Castigat', 'file_type': 'xml', 'access_right': 'open_public'}],
 'Croquis_vaudois': ['Croquis_vaudois',
  {'alias': 'Croquis', 'file_type': 'xml', 'access_right': 'open_public'}],
 'Courrier_du_Léman': ['Courrier_du_Léman', None],
 "Feuille_d'avis_de_Moudon": ["Feuille_d'avis_de_Moudon",
  {'alias': 'FAMDE', 'file_type': 'json', 'access_right': 'open_public'}],
 "Feuille_d'avis_de_Nyon": ["Feuille_d'avis_de_Nyon",
  {'alias': 'FAN', 'file_type': 'json', 'access_right': 'open_public'}],
 "Feuille_d'avis_de_Payerne,_Moudon_et_Avenches": ["Feuille_d'avis_de_Payerne,_Moudon_et_Avenches",
  {'alias': 'feuillePMA', 'file_type': 'json', 'access_right': 'open_public'}],
 "Feuille_d'avis_de_Payerne": ["Feuille_d'avis_de_Payerne",
  {'alias': 'feuilleP', 'file_type': 'json', 'access_right': 'open_pub

In [40]:
def find_LCS(s1, s2):
    res = pylcs.lcs_string_idx(s1, s2)
    return ''.join([s2[i] for i in res if i != -1])

In [41]:
for k, v in ar_and_alias.items():
    print(f"-{k}:")
    v['matches'] = []
    v['lcs'] = {}
    for mk, mv in missing_aliases.items():
        if mv[1] is not None:
            if mv[1]['alias'] == v['alias']:
                print(f'  title added to {k}', mk)
                v['title'] = mk
                del v['lcs']
                del v['matches']
                break
        else:
            lcs = find_LCS(k, mk)
            v['lcs'][mk] = lcs
            if k.split('_')[0] == mk.split('_')[0]:
                print(f"  Potential match: {k} and {mk}")
                v['matches'].append(mk)
                if k[-2] == mk[-2]:
                    print(f"  ! VERY Potential match: {k} and {mk}")
                    v['matches'] = [mk]
            elif k.split('_')[-1] == mk.split('_')[-1]:
                print(f"  Potential match: {k} and {mk}")
                v['matches'].append(mk)

    if 'matches' in v and len(v['matches']) == 1:
        print(f"will add title {v['matches']} to {k}")
        v['title'] = v['matches'][0]
        del v['lcs']
        del v['matches']
    elif 'lcs' in v:
        max_l = 0
        title = ''
        for m in v['matches']:
            if len(v['lcs'][m]) > max_l:
                max_l = len(v['lcs'][m])
                title = m
        print(f"will add title {title} to {k}")
        v['title'] = title
        del v['lcs']
        del v['matches']

            #print(f'title missing for {k}', mk, str(mk), str(k))

-Mercure_suisse:
  title added to Mercure_suisse Mercure_suisse
-Nouvelliste_suisse:
  title added to Nouvelliste_suisse Nouvelliste_suisse
-Journal_helvÃ©tique:
  Potential match: Journal_helvÃ©tique and Journal_de_vevey_et_des_étrangers
  Potential match: Journal_helvÃ©tique and Journal_helvétique
  ! VERY Potential match: Journal_helvÃ©tique and Journal_helvétique
will add title ['Journal_helvétique'] to Journal_helvÃ©tique
-Messager_boÃ®teux:
  Potential match: Messager_boÃ®teux and Messager_boîteux
  ! VERY Potential match: Messager_boÃ®teux and Messager_boîteux
will add title ['Messager_boîteux'] to Messager_boÃ®teux
-Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud:
  Potential match: Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud and L’ami_de_la_vérité__journal_du_Canton_de_Vaud
will add title ['L’ami_de_la_vérité__journal_du_Canton_de_Vaud'] to Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud
-Almanach_pour_le_commerce:
  title added to Almanach_pour_le_commerce Almanach_

In [43]:
new_ar_and_alias = {}
for k, v in ar_and_alias.items():
    title = v['title']
    del v['title']
    v['bin_title'] = k
    new_ar_and_alias[title] = v

In [45]:
access_rights_and_aliases_file = '/home/piconti/impresso-text-acquisition/text_importer/data/sample_data/BCUL/access_rights_and_aliases.json'

with open(access_rights_and_aliases_file, 'w', encoding='utf-8') as f_out:
    json.dump(new_ar_and_alias, f_out, ensure_ascii=False, indent=4)

In [42]:
ar_and_alias

{'Mercure_suisse': {'alias': 'ME',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'Mercure_suisse'},
 'Nouvelliste_suisse': {'alias': 'NS',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'Nouvelliste_suisse'},
 'Journal_helvÃ©tique': {'alias': 'JH',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'Journal_helvétique'},
 'Messager_boÃ®teux': {'alias': 'MB',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'Messager_boîteux'},
 'Lâ\x80\x99ami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud': {'alias': 'NV',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'L’ami_de_la_vérité__journal_du_Canton_de_Vaud'},
 'Almanach_pour_le_commerce': {'alias': 'ACI',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'Almanach_pour_le_commerce'},
 'Le_PÃ¨re_JÃ©rÃ´me': {'alias': 'PJ',
  'file_type': 'xml',
  'access_right': 'open_public',
  'title': 'Le_Père_Jérôme'},
 'Le_Charivari_[1]': {'alias': 'Charivar

In [5]:
for j_dir in os.listdir(base_dir): 
    if j_dir not in ['OLD', 'wrong_BCUL', '.DS_Store']:
        if j_dir in ar_and_alias:
            print(j_dir, "OK: ", ar_and_alias[j_dir])
        else:
            print(j_dir, 'NOT OK!')

Almanach_pour_le_commerce OK:  {'alias': 'ACI', 'file_type': 'xml', 'access_right': 'open_public'}
Castigat_ridendo_mores OK:  {'alias': 'Castigat', 'file_type': 'xml', 'access_right': 'open_public'}
Croquis_vaudois OK:  {'alias': 'Croquis', 'file_type': 'xml', 'access_right': 'open_public'}
Courrier_du_LÃ©man OK:  {'alias': 'CL', 'file_type': 'json', 'access_right': 'open_public'}
Feuille_d'avis_de_Moudon OK:  {'alias': 'FAMDE', 'file_type': 'json', 'access_right': 'open_public'}
Feuille_d'avis_de_Nyon OK:  {'alias': 'FAN', 'file_type': 'json', 'access_right': 'open_public'}
Feuille_d'avis_de_Payerne,_Moudon_et_Avenches OK:  {'alias': 'feuillePMA', 'file_type': 'json', 'access_right': 'open_public'}
Feuille_d'avis_de_Payerne OK:  {'alias': 'feuilleP', 'file_type': 'json', 'access_right': 'open_public'}
Gazette_des_Villages OK:  {'alias': 'GAVi', 'file_type': 'json', 'access_right': 'open_public'}
Indicateur_de_Lausanne OK:  {'alias': 'AV', 'file_type': 'xml', 'access_right': 'open_pub

In [4]:
all([name in dirs for name in batch_1_titles['unicode_name_in_fs'] if name is not None])

NameError: name 'batch_1_titles' is not defined

In [9]:
def parse_date(mit_filename: str) -> tuple[date, str]:
    """Given the Mit filename, parse the Journal name and the date.

    Args:
        mit_filename (str): Filename of the 'mit' file.

    Returns:
        Tuple[date, str]: Publication date of the issue, and journal name.
    """
    split = mit_filename.split('/')
    # extract date of publication
    year, month, day = int(split[-5]), int(split[-4]), int(split[-3])
    #normalize the month & day values if they are out of range
    month = max(min(month, 12), 1)
    day = max(min(day, 31), 1)

    # extract journal name alias
    #basename = os.path.splitext(split[-1])[0]
    #journal_alias = replace_alias(basename.split('_')[0], split[-6])
    
    return datetime(year, month, day).date() #, journal_alias

In [25]:
from collections import namedtuple
BculIssueDir = namedtuple(
    "IssueDirectory", ["journal", "date", "edition", "path", "rights", "mit_file_type"]
)

In [26]:
def dir2issue(path: str, journal_info: dict[str, str]) -> BculIssueDir:
    """Create a `BculIssueDir` object from a directory.

    Note:
        This function is called internally by `detect_issues`

    Args:
        path (str): The path of the issue.
        access_rights (dict): Dictionary for access rights.

    Returns:
        Optional[BculIssueDir]: New `BculIssueDir` object.
    """
    mit_file = find_mit_file(path)
    if mit_file is None:
        logger.error("Could not find MIT file in %s", path)
        return None

    if not mit_file.endswith(journal_info["file_type"]):
        logger.warning(
            "Found mit file %s does not correspond to mit file type %s",
            path,
            journal_info["file_type"],
        )
        # override the mit file type if the extension of the file found does not match
        journal_info["file_type"] = mit_file.split(".")[-1]

    date = parse_date(mit_file)

    return BculIssueDir(
        journal=journal_info["alias"],
        date=date,
        edition="a",
        path=path,
        rights=journal_info["access_right"],
        mit_file_type=journal_info["file_type"],
    )

In [19]:
ar_and_alias.keys()

dict_keys(['Mercure_suisse', 'Nouvelliste_suisse', 'Journal_helvÃ©tique', 'Messager_boÃ®teux', 'Lâ\x80\x99ami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud', 'Almanach_pour_le_commerce', 'Le_PÃ¨re_JÃ©rÃ´me', 'Le_Charivari_[1]', 'Le_Phare_de_Nyon', 'Le_Phare_du_LÃ©man', "Journal_d'Yverdon_2", "Feuille_d'avis_de_Payerne", 'Le_Grelot', 'Le_peuple_souverain', "Feuille_d'avis_de_Payerne,_Moudon_et_Avenches", 'La_GuÃªpe_[1]', "Feuille_d'avis_de_Moudon", "L'Observateur_du_LÃ©man", 'La_Griffe', "Feuille_d'avis_de_Nyon", 'Gazette_des_Villages', 'La_Fronde', 'Le_Charivari_[2]', 'Castigat_ridendo_mores', 'Le_Moniteur', 'Le_messager', 'Le_Ouistiti', 'Le_Touriste__La_Suisse_illustrÃ©e', 'Journal_de_Vevey', 'Indicateur_de_Lausanne', 'Courrier_du_LÃ©man', 'Croquis_vaudois', 'Le_Moustique_(Vevey)', 'La_Bombe', 'Le_Veveysan_1', 'Journal_de_vevey_et_des_Ã©trangers', 'La_GuÃªpe_[2]', 'La_Cancoire', 'La_revue_agricole'])

In [20]:
ar_and_alias['Courrier_du_LÃ©man']

{'alias': 'CL', 'file_type': 'json', 'access_right': 'open_public'}

In [33]:
dir_path, dirs, files = next(os.walk(base_dir))
    
journal_dirs = [os.path.join(dir_path, _dir) 
                for _dir in dirs 
                if _dir not in ['OLD', 'wrong_BCUL'] and _dir in ar_and_alias]

issue_dirs = []
for journal in journal_dirs:
    #journal_info = titles_as_dict[journal.split('/')[-1]]
    #logger.info(f"Detecting issues for {journal}.")
    for dir_path, dirs, files in os.walk(journal):
        if len(files)>1 and 'solr' not in dir_path:
            title = dir_path.replace(base_dir, '').split('/')[0]
            if title != journal.split('/')[-1]:
                print(title, journal)
            if title not in ar_and_alias:
                print(dir_path, title, title in ar_and_alias)
            else:
                print(dir2issue(dir_path, ar_and_alias[title]))
                issue_dirs.append((dir_path, ar_and_alias[title]))
            break

issues = []
for _dir in issue_dirs:
    i_dir= dir2issue(_dir)
    print(i_dir)
    issues.append(i_dir)
#issues = [dir2issue(_dir) for _dir in issue_dirs]

IssueDirectory(journal='ACI', date=datetime.date(1832, 1, 1), edition='a', path='/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce/1832/01/01/171722', rights='open_public', mit_file_type='xml')
IssueDirectory(journal='Castigat', date=datetime.date(1879, 11, 1), edition='a', path='/mnt/project_impresso/original/BCUL/Castigat_ridendo_mores/1879/11/01/170474', rights='open_public', mit_file_type='xml')
IssueDirectory(journal='Croquis', date=datetime.date(1884, 3, 1), edition='a', path='/mnt/project_impresso/original/BCUL/Croquis_vaudois/1884/03/01/126485', rights='open_public', mit_file_type='xml')
IssueDirectory(journal='CL', date=datetime.date(1882, 9, 30), edition='a', path='/mnt/project_impresso/original/BCUL/Courrier_du_LÃ©man/1882/09/30/257141', rights='open_public', mit_file_type='json')
IssueDirectory(journal='FAMDE', date=datetime.date(1859, 2, 5), edition='a', path="/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Moudon/1859/02/05/539294", rights='open_public',

TypeError: dir2issue() missing 1 required positional argument: 'journal_info'

In [ ]:
def alias_and_mit_file_type(files):
    for f in files:
        if 'mit' in f:
            return f.split('.')[-1], f.split('_')[0]

In [ ]:
def _detect_issues(base_dir: str, access_rights: str, issue_dirs, mit_file_nature, aliases) -> list[BculIssueDir]:
    """Detect BCUL newspaper issues to import within the filesystem.

    This function expects the directory structure that BCUL used to
    organize the dump of Abbyy files.

    Args:
        base_dir (str): Path to the base directory of newspaper data.
        access_rights (str): Path to ``access_rights.json`` file.

    Returns:
        list[BculIssueDir]: List of `BCULIssueDir` instances, to be imported.
    """
    
    dir_path, dirs, files = next(os.walk(base_dir))
    
    journal_dirs = [os.path.join(dir_path, _dir) for _dir in dirs if _dir not in ['OLD', 'wrong_BCUL']]

    #mit_file_nature = {}
    #aliases = {}
    for journal in journal_dirs:
        print(f"\n Detecting issues for {journal}.")
        for dir_path, dirs, files in os.walk(journal):
            if len(files)>1 and 'solr' not in dir_path:
                issue_dirs.append(dir_path)
                print(dir_path)
                if journal not in aliases:
                    mit_file_nature[journal], aliases[journal] = alias_and_mit_file_type(files)
                    #aliases[journal] = replace_alias(aliases[journal], journal)

    return mit_file_nature, aliases, [dir2issue(_dir, None) for _dir in issue_dirs]

In [ ]:
issue_dirs = []
mit_file_nature, aliases = {}, {}
mit_file_nature, aliases, issues = _detect_issues(base_dir, '', issue_dirs, mit_file_nature, aliases)

In [ ]:
mit_file_nature

In [ ]:
found_aliases = {
 '/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce': 'ACI',
 '/mnt/project_impresso/original/BCUL/Castigat_ridendo_mores': 'Castigat',
 '/mnt/project_impresso/original/BCUL/Croquis_vaudois': 'Croquis',
 '/mnt/project_impresso/original/BCUL/Courrier_du_LÃ©man': 'CL',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Moudon": 'FAMDE',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Nyon": 'FAN',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Payerne,_Moudon_et_Avenches": 'feuille',
 "/mnt/project_impresso/original/BCUL/Feuille_d'avis_de_Payerne": 'feuille',
 '/mnt/project_impresso/original/BCUL/Gazette_des_Villages': 'GAV',
 '/mnt/project_impresso/original/BCUL/Indicateur_de_Lausanne': 'AV',
 "/mnt/project_impresso/original/BCUL/Journal_d'Yverdon_2": 'JY2',
 '/mnt/project_impresso/original/BCUL/Journal_de_Vevey': 'JV'
}

In [ ]:
aliases_json= {
    "Courrier_du_LÃ©man": 'CL',
    "Feuille_d'avis_de_Moudon": 'FAMDE',
    "Feuille_d'avis_de_Nyon": 'FAN',
    "Feuille_d'avis_de_Payerne,_Moudon_et_Avenches": "feuillePMA", #'feuille' in fs
    "Feuille_d'avis_de_Payerne": "feuilleP", # 'feuille' in fs
    "Gazette_des_Villages": "GAVi", # GAV
    "Journal_d'Yverdon_2": 'JY2', 
    "Journal_de_Vevey": 'JV',
    "Journal_de_vevey_et_des_Ã©trangers": 'JVE',
    "L'Observateur_du_LÃ©man": 'OBS',
    "Le_Moustique_(Vevey)": 'Moustique', 
    "Le_Phare_de_Nyon": 'PDN', 
    "Le_Phare_du_LÃ©man": 'PDL', # FAN in fs
    "Le_Touriste__La_Suisse_illustrÃ©e": "TouSuIl", #'TO-SU-IL' in fs
    "Le_Veveysan_1": 'VVS1', # VVS also for La_Veveysanne_–_La_Patrie/La_Veveysanne 
    "Le_messager": 'MESSAGER',
    "Le_peuple_souverain": 'PS'
}

In [ ]:
len(issue_dirs)

## Debug functions in helpers

In [2]:
from text_importer.importers.bcul.helpers import parse_textblock

In [3]:
page_xml_file = '/home/piconti/impresso-text-acquisition/text_importer/data/sample_data/BCUL/388793/6254688.xml'

with open(page_xml_file, encoding='utf-8') as f:
    xml_doc = BeautifulSoup(f, "xml")

xml_doc

<?xml version="1.0" encoding="utf-8"?>
<page height="5187" originalCoords="1" resolution="300" width="3629">
<block b="198" blockType="Text" l="111" r="409" t="152">
<region>
<rect b="198" l="111" r="409" t="152"/>
</region>
<text>
<par lineSpacing="1160">
<line b="191" baseline="191" l="117" r="403" t="158">
<formatting ff="Times New Roman" fs="10.5" lang="FrenchStandard">
<charParams b="191" charConfidence="100" l="117" meanStrokeWidth="37" r="136" serifProbability="255" t="159" wordFirst="1" wordFromDictionary="0" wordIdentifier="0" wordLeftMost="1" wordNormal="0" wordNumeric="0" wordPenalty="5">7</charParams>
<charParams b="191" charConfidence="100" l="140" meanStrokeWidth="37" r="159" serifProbability="255" t="159" wordFromDictionary="0" wordIdentifier="0" wordNormal="0" wordNumeric="0" wordPenalty="5">7</charParams>
</formatting>
<formatting ff="Times New Roman" fs="10.5" lang="FrenchStandard" superscript="1">
<charParams b="177" charConfidence="44" l="161" meanStrokeWidth="37" r

In [4]:
text_blocks = xml_doc.findAll("block", {"blockType": "Text"})
page_data = [parse_textblock(tb, "FAM-1937-01-30-a-i0001") for tb in text_blocks]
page_data

inside parse_char_tokens: [<charParams b="191" charConfidence="100" l="117" meanStrokeWidth="37" r="136" serifProbability="255" t="159" wordFirst="1" wordFromDictionary="0" wordIdentifier="0" wordLeftMost="1" wordNormal="0" wordNumeric="0" wordPenalty="5">7</charParams>, <charParams b="191" charConfidence="100" l="140" meanStrokeWidth="37" r="159" serifProbability="255" t="159" wordFromDictionary="0" wordIdentifier="0" wordNormal="0" wordNumeric="0" wordPenalty="5">7</charParams>, <charParams b="177" charConfidence="44" l="161" meanStrokeWidth="37" r="182" serifProbability="100" suspicious="1" t="164" wordFromDictionary="0" wordIdentifier="0" wordNormal="0" wordNumeric="0" wordPenalty="5">m</charParams>, <charParams b="177" charConfidence="24" l="186" meanStrokeWidth="37" r="197" serifProbability="77" suspicious="1" t="164" wordFromDictionary="0" wordIdentifier="0" wordNormal="0" wordNumeric="0" wordPenalty="5">c</charParams>, <charParams b="191" l="198" r="242" t="159"> </charParams>,

[{'c': [111, 152, 298, 46],
  'p': [{'c': [111, 152, 298, 46],
    'l': [{'c': [117, 158, 286, 33],
      't': [{'c': [117, 159, 80, 18], 'tx': '77mc'},
       {'c': [243, 159, 160, 32], 'tx': 'ANNEE'}]}]}],
  'pOf': 'FAM-1937-01-30-a-i0001'},
 {'c': [1723, 151, 109, 46],
  'p': [{'c': [1723, 151, 109, 46],
    'l': [{'c': [1729, 157, 97, 34],
      't': [{'c': [1729, 157, 46, 18], 'tx': 'N°'},
       {'c': [1806, 159, 20, 32], 'tx': '9'}]}]}],
  'pOf': 'FAM-1937-01-30-a-i0001'},
 {'c': [2831, 153, 652, 48],
  'p': [{'c': [2831, 153, 652, 48],
    'l': [{'c': [2837, 159, 640, 36],
      't': [{'c': [2837, 160, 173, 33], 'tx': 'SAMEDI'},
       {'c': [3057, 161, 44, 33], 'tx': '30'},
       {'c': [3147, 161, 195, 33], 'tx': 'JANVIER'},
       {'c': [3392, 162, 85, 33], 'tx': '1937'}]}]}],
  'pOf': 'FAM-1937-01-30-a-i0001'},
 {'c': [234, 278, 280, 110],
  'p': [{'c': [234, 278, 280, 110],
    'l': [{'c': [251, 297, 256, 91],
      't': [{'c': [251, 307, 106, 81], 'tx': 'W'},
       {'c':

In [5]:
example_2 = '/home/piconti/impresso-text-acquisition/text_importer/data/sample_data/BCUL/171722/ACI_1832_00_00_0001_page_1.xml'

with open(example_2, encoding='utf-8') as f:
    xml_doc_2 = BeautifulSoup(f, "xml")

text_blocks = xml_doc_2.findAll("block", {"blockType": "Text"})
page_data = [parse_textblock(tb, "ACI-1832-01-01-a-i0001") for tb in text_blocks]
page_data

inside parse_char_tokens: [<charParams b="1526" charConfidence="98" l="359" meanStrokeWidth="32" r="395" serifProbability="81" t="1486" wordFromDictionary="1" wordIdentifier="0" wordNormal="1" wordNumeric="0" wordPenalty="0" wordStart="1">L</charParams>, <charParams b="1526" charConfidence="83" l="400" meanStrokeWidth="32" r="437" serifProbability="98" t="1485" wordFromDictionary="1" wordIdentifier="0" wordNormal="1" wordNumeric="0" wordPenalty="0" wordStart="0">A</charParams>, <charParams b="1525" charConfidence="80" l="442" meanStrokeWidth="32" r="482" serifProbability="100" t="1483" wordFromDictionary="1" wordIdentifier="0" wordNormal="1" wordNumeric="0" wordPenalty="0" wordStart="0">U</charParams>, <charParams b="1523" charConfidence="92" l="488" meanStrokeWidth="32" r="521" serifProbability="73" t="1482" wordFromDictionary="1" wordIdentifier="0" wordNormal="1" wordNumeric="0" wordPenalty="0" wordStart="0">S</charParams>, <charParams b="1521" charConfidence="94" l="525" meanStrokeW

[{'c': [352, 1471, 350, 61],
  'p': [{'c': [352, 1471, 350, 61],
    'l': [{'c': [359, 1478, 336, 48],
      't': [{'c': [359, 1486, 336, 33], 'tx': 'LAUSANNE'}]}]}],
  'pOf': 'ACI-1832-01-01-a-i0001'},
 {'c': [153, 1552, 772, 51],
  'p': [{'c': [153, 1552, 772, 51],
    'l': [{'c': [160, 1556, 759, 40],
      't': [{'c': [160, 1571, 41, 25], 'tx': 'AU'},
       {'c': [216, 1570, 65, 25], 'tx': 'BAZ'},
       {'c': [286, 1571, 41, 22], 'tx': 'AR'},
       {'c': [341, 1567, 150, 23], 'tx': 'VAUDOIS'},
       {'c': [502, 1585, 5, 9], 'tx': ','},
       {'c': [530, 1565, 43, 23], 'tx': 'AU'},
       {'c': [585, 1561, 238, 23], 'tx': 'CHEMIN-NEUF'},
       {'c': [832, 1580, 4, 10], 'tx': ','},
       {'c': [855, 1558, 38, 16], 'tx': 'N°'},
       {'c': [904, 1556, 15, 35], 'tx': '4'}]}]}],
  'pOf': 'ACI-1832-01-01-a-i0001'},
 {'c': [162, 206, 172, 56],
  'p': [{'c': [162, 206, 172, 56],
    'l': [{'c': [163, 207, 174, 56],
      't': [{'c': [163, 211, 140, 46], 'tx': 'us'},
       {'c': [3

In [6]:
example_3 = '/home/piconti/impresso-text-acquisition/text_importer/data/sample_data/BCUL/46165/FAL_1762_12_07_0001_page_1.xml'

with open(example_3, encoding='utf-8') as f:
    xml_doc_3 = BeautifulSoup(f, "xml")

text_blocks = xml_doc_3.findAll("block", {"blockType": "Text"})
page_data = [parse_textblock(tb, "ACI-1832-01-01-a-i0001") for tb in text_blocks]
page_data

inside parse_char_tokens: [<charParams b="147" charConfidence="34" l="925" meanStrokeWidth="44" r="941" serifProbability="255" t="99" wordFromDictionary="false" wordIdentifier="true" wordNormal="false" wordNumeric="false" wordPenalty="0" wordStart="true">)</charParams>, <charParams b="144" charConfidence="25" l="951" meanStrokeWidth="44" r="964" serifProbability="255" t="100" wordFromDictionary="false" wordIdentifier="true" wordNormal="false" wordNumeric="false" wordPenalty="0" wordStart="false">(</charParams>, <charParams b="145" l="964" r="979" t="100"> </charParams>, <charParams b="145" l="979" r="994" t="100"> </charParams>, <charParams b="145" l="994" r="1009" t="100"> </charParams>, <charParams b="145" l="1009" r="1024" t="100"> </charParams>, <charParams b="145" charConfidence="25" l="1024" meanStrokeWidth="39" r="1046" serifProbability="74" t="110" wordFromDictionary="false" wordIdentifier="false" wordNormal="false" wordNumeric="true" wordPenalty="2" wordStart="true">9</charPar

[{'c': [306, 89, 1485, 710],
  'p': [{'c': [306, 89, 1485, 710],
    'l': [{'c': [925, 92, 261, 55],
      't': [{'c': [925, 99, 39, 45], 'tx': ')('},
       {'c': [1024, 110, 52, 33], 'tx': '91'},
       {'c': [1147, 92, 39, 50], 'tx': 'X'}]},
     {'c': [320, 336, 1459, 106],
      't': [{'c': [320, 354, 93, 88], 'tx': 'A'},
       {'c': [519, 348, 94, 93], 'tx': 'N'},
       {'c': [722, 347, 92, 90], 'tx': 'N'},
       {'c': [922, 344, 87, 90], 'tx': 'O'},
       {'c': [1116, 338, 92, 91], 'tx': 'N'},
       {'c': [1317, 340, 83, 89], 'tx': 'G'},
       {'c': [1506, 336, 75, 92], 'tx': 'E'},
       {'c': [1684, 338, 95, 97], 'tx': 'S,'}]},
     {'c': [608, 474, 878, 61],
      't': [{'c': [608, 485, 110, 48], 'tx': 'ET'},
       {'c': [805, 483, 220, 47], 'tx': 'AVIS'},
       {'c': [1109, 477, 377, 51], 'tx': 'DIVERS.'}]},
     {'c': [494, 549, 1131, 64],
      't': [{'c': [494, 575, 273, 36], 'tx': 'XXIVme.'},
       {'c': [828, 571, 286, 35], 'tx': 'FEUILLE'},
       {'c': [1178,

In [7]:
int('True')

ValueError: invalid literal for int() with base 10: 'True'